# AI Signalwire demo

Run the cells top to bottom. The pipeline fetches AI news and framework releases, checks relevance, drafts posts, validates and evaluates them, then waits for your approval.

## Setup

In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())
from dotenv import load_dotenv
load_dotenv()
import pandas as pd
from IPython.display import HTML, display
import ipywidgets as widgets
from src.config import load_settings, load_sources
from src.llm import get_llm
from src.storage.db import Database
from src.pipeline import run_pipeline
from src.review.queue import pending_review, needs_attention, approve, reject
from src.publishing.local_blog import render_blog_html

settings = load_settings()
db = Database(settings.db_path)
llm = get_llm(settings, 'agent')
eval_llm = get_llm(settings, 'evaluator')
mode = 'fake model' if settings.fake_llm else f'{settings.provider} ({settings.agent_model}, evaluator {settings.evaluator_model})'
print('LLM:', mode)
print('Articles:', 'canned demo set' if settings.demo_mode else 'live sources')


## Configure

Sources come from `config/sources.yaml`. Disable any you don't want there.

In [ ]:
sources = load_sources()
print(f'{len(sources)} sources, up to {settings.items_per_source} items each')
pd.DataFrame([s.model_dump() for s in sources])[['name', 'type', 'adapter', 'url']]


## Run pipeline

In [ ]:
result = run_pipeline(db, settings, sources, llm, eval_llm)
print({k: v for k, v in result.items() if k != 'article_ids'})
def status_table():
    rows = db.list_articles()
    return pd.DataFrame([{'id': r['id'], 'source': r['source_name'], 'title': r['title'][:70], 'status': r['status']} for r in rows])
status_table()


## Inspect one article

In [ ]:
def show(article_id):
    r = db.get(article_id)
    print(f"#{r['id']} {r['title']}  [{r['status']}]")
    print('Relevance:', r['relevance'])
    print('Draft:', r['post'])
    print('Validation errors:', r['validation'])
    print('Evaluator:', r['evaluation'])
    print('Error:', r['error'])
rows = pending_review(db) or db.list_articles()
if rows:
    show(rows[0]['id'])


## Review queue

Approve publishes to the local blog. Reject sends the item back for investigation.

In [ ]:
blog_out = widgets.Output()
def refresh_blog():
    with blog_out:
        blog_out.clear_output()
        display(HTML(render_blog_html(db)))

def queue_row(r):
    post = r['post']
    label = widgets.HTML(f"<b>{post['title']}</b><br>Evaluator: {r['evaluation']['verdict']}, {r['evaluation']['reason']}")
    ok, no = widgets.Button(description='Approve', button_style='success'), widgets.Button(description='Reject', button_style='danger')
    def done(text):
        ok.disabled = no.disabled = True
        label.value += f'<br><i>{text}</i>'
        refresh_blog()
    ok.on_click(lambda b: (approve(db, r['id']), done('Approved and published')))
    no.on_click(lambda b: (reject(db, r['id'], 'Rejected in review'), done('Rejected')))
    return widgets.HBox([label, ok, no])

pending = pending_review(db)
display(widgets.VBox([queue_row(r) for r in pending]) if pending else widgets.HTML('No drafts waiting for review.'))
print('Needs attention:')
pd.DataFrame([{'id': r['id'], 'title': r['title'][:60], 'status': r['status'], 'reason': (r['error'] or '')[:80]} for r in needs_attention(db)])


## Blog

Published posts appear here. Run this cell again after approving.

In [ ]:
refresh_blog()
blog_out


## Reset

Uncomment to clear the database for a fresh demo.

In [ ]:
# db.reset()
